#Teach an LLM to Spell with Group Relative Policy Optimization (GRPO)


Large language models (LLMs) are bad at spelling. This is because tokenizers break words into smaller pieces, so the model learns about sub-word units rather than whole words and their spellings.

So, I'll use Group Relative Policy Optimization (GRPO) and a technique called Parameter-Efficient Fine-Tuning (PEFT) with Low-Rank Adaptation (LoRA) to teach a small LLM how to spell words. This is a classic example of teaching a model a new skill that isn't well-represented in its pre-training data.

---
## 1- Setup

In [5]:
import os
import torch
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig


if torch.cuda.is_available():
    device = torch.device("cuda")  # NVIDIA GPU
else:
    device = torch.device("cpu")
torch.set_num_threads(max(1, os.cpu_count() // 2))
print("Using device:", device)

Using device: cuda


## 2-  Load the tokenizer and base model

The model HuggingFaceTB/SmolLM2-135M-Instruct is a small, instruction-tuned model. It has 135 million parameters, making it lightweight and efficient for fine-tuning. It's not the most powerful model, but it's a good choice for demonstrating the concepts of SFT and PEFT with LoRA, especially on a CPU or limited GPU resources.

In [6]:
model_id = "HuggingFaceTB/SmolLM2-135M-Instruct"

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Load the model
model = AutoModelForCausalLM.from_pretrained(
    model_id,
)

# Copy the model to the device (GPU, MPS, or CPU)
model = model.to(device)
# <<< END SOLUTION SECTION

print("Model parameters (total):", sum(p.numel() for p in model.parameters()))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/861 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Model parameters (total): 134515008


## 3- Create the dataset

In [7]:
ALL_WORDS = [
    "idea", "glow", "rust", "maze", "echo", "wisp", "veto", "lush", "gaze", "knit", "fume", "plow",
    "void", "oath", "grim", "crisp", "lunar", "fable", "quest", "verge", "brawn", "elude", "aisle",
    "ember", "crave", "ivory", "mirth", "knack", "wryly", "onset", "mosaic", "velvet", "sphinx",
    "radius", "summit", "banner", "cipher", "glisten", "mantle", "scarab", "expose", "fathom",
    "tavern", "fusion", "relish", "lantern", "enchant", "torrent", "capture", "orchard", "eclipse",
    "frescos", "triumph", "absolve", "gossipy", "prelude", "whistle", "resolve", "zealous",
    "mirage", "aperture", "sapphire",
]

In [8]:
def generate_records():
    for word in ALL_WORDS:
        yield {
            # We will use the GRPOTrainer which expects to receieve formatted prompts
            # https://huggingface.co/docs/trl/main/en/grpo_trainer
            "prompt": (
                f"You spell words with hyphens between the letters like this W-O-R-D.\nWord:\n{word}\n\n"
                + "Spelling:\n"
            ),
            "completion": "-".join(word).upper() + ".",
            '''
            GRPOTrainer does not expect a completion, but we can add extra columns to our dataset
            that our reward functions will use to grade the completions provided by the LLM.
            '''
            "word": word,
            "spelling": "-".join(word).upper(),
        }


ds = Dataset.from_generator(generate_records)

ds = ds.train_test_split(test_size=0.1, seed=42)

# Show the first item of the train split
ds["train"][0]

Generating train split: 0 examples [00:00, ? examples/s]

{'prompt': 'You spell words with hyphens between the letters like this W-O-R-D.\nWord:\ntriumph\n\nSpelling:\n',
 'completion': 'T-R-I-U-M-P-H.',
 'word': 'triumph',
 'spelling': 'T-R-I-U-M-P-H'}

## 4- Evaluate the base model

Before we fine-tune the model, let's see how it performs on the spelling task. We'll create a helper function to generate a spelling for a given word and compare it to the correct answer.


In [9]:
def check_spelling(
    model, tokenizer, prompt: str, actual_spelling: str, max_new_tokens: int = 20
) -> (str, str):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    gen = model.generate(
        **inputs, max_new_tokens=max_new_tokens, use_cache=False
    )  # No parameters = greedy search
    output = tokenizer.decode(gen[0], skip_special_tokens=True)

    # Extract the generated spelling from the full output string
    proposed_spelling = output.split("Spelling:\n")[-1].strip().split("\n")[0].strip()

    # strip any whitepsace from the actual spelling
    actual_spelling = actual_spelling.strip()

    print(
        f"Proposed: {proposed_spelling} | Actual: {actual_spelling} "
        f"| Matches: {'✅' if proposed_spelling == actual_spelling else '❌'}"
    )

    # Remove hyphens for a character-by-character comparison
    proposed_spelling = proposed_spelling.replace("-", "")
    actual_spelling = actual_spelling.replace("-", "")

    # Calculate the proportion of the spelling that was correct
    num_correct = sum(1 for a, b in zip(actual_spelling, proposed_spelling) if a == b)

    return num_correct / len(actual_spelling)  # Return proportion correct

In [10]:
proportion_correct = 0.0

for example in ds["train"].select(range(20)):
    prompt = example["prompt"]
    spelling = example["spelling"]
    result = check_spelling(
        model=model,
        tokenizer=tokenizer,
        prompt=prompt,
        actual_spelling=spelling,
        max_new_tokens=20,
    )
    proportion_correct += result

print(f"{proportion_correct}/20.0 words correct")

Proposed: trium | Actual: T-R-I-U-M-P-H | Matches: ❌
Proposed: sapp | Actual: S-A-P-P-H-I-R-E | Matches: ❌
Proposed: expose | Actual: E-X-P-O-S-E | Matches: ❌
Proposed: fres | Actual: F-R-E-S-C-O-S | Matches: ❌
Proposed: wisp | Actual: W-I-S-P | Matches: ❌
Proposed: mi-er-ge | Actual: M-I-R-A-G-E | Matches: ❌
Proposed: ivory | Actual: I-V-O-R-Y | Matches: ❌
Proposed: onset | Actual: O-N-S-E-T | Matches: ❌
Proposed: elude | Actual: E-L-U-D-E | Matches: ❌
Proposed: sphinx | Actual: S-P-H-I-N-X | Matches: ❌
Proposed: brawn | Actual: B-R-A-W-N | Matches: ❌
Proposed: goss | Actual: G-O-S-S-I-P-Y | Matches: ❌
Proposed: enchant | Actual: E-N-C-H-A-N-T | Matches: ❌
Proposed: tavern | Actual: T-A-V-E-R-N | Matches: ❌
Proposed: whistle | Actual: W-H-I-S-T-L-E | Matches: ❌
Proposed: W-O-R-D | Actual: C-A-P-T-U-R-E | Matches: ❌
Proposed: echo | Actual: E-C-H-O | Matches: ❌
Proposed: mirth | Actual: M-I-R-T-H | Matches: ❌
Proposed: cris | Actual: C-R-I-S-P | Matches: ❌
Proposed: zeal | Actual: Z-E-

As you can see, the base model is terrible at spelling. It mostly just repeats the word back.

## 5- Configure LoRA and train the model

 We use a LoRA config so only a tiny fraction of parameters are trainable.

In [11]:
# Print how many params are trainable at first
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(
    f"Trainable params BEFORE: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)"
)


lora_config = LoraConfig(
    r=64,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)

# Print the number of trainable parameters after applying LoRA
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(
    f"Trainable params AFTER: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)"
)


Trainable params BEFORE: 134,515,008 / 134,515,008 (100.00%)
Trainable params AFTER: 3,686,400 / 138,201,408 (2.67%)


Now let’s set the training arguments. We'll use SFTConfig from the TRL library, which is a wrapper around the standard Training Arguments. We keep epochs, batch size, and sequence length modest to finish training quickly.

In [12]:
output_dir = "data/model"

training_args = SFTConfig(
    output_dir=output_dir,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    learning_rate=5 * 1e-4,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=20,
    save_strategy="no",
    report_to=[],
    fp16=False,
    lr_scheduler_type="cosine",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    args=training_args,
)
trainer.train()

proportion_correct = 0.0

for example in ds["train"].select(range(20)):
    prompt = example["prompt"]
    spelling = example["spelling"]
    result = check_spelling(
        model=model,
        tokenizer=tokenizer,
        prompt=prompt,
        actual_spelling=spelling,
        max_new_tokens=20,
    )
    proportion_correct += result

print(f"{proportion_correct}/20.0 words correct")

Adding EOS to train dataset:   0%|          | 0/55 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/55 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/55 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/7 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/7 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/7 [00:00<?, ? examples/s]

Step,Training Loss,Validation Loss
20,1.079252,0.865687
40,0.529107,0.739169
60,0.428893,0.728531


Proposed: T-I-R-U-M-P-H. | Actual: T-R-I-U-M-P-H | Matches: ❌
Proposed: S-A-P-I-C-H. | Actual: S-A-P-P-H-I-R-E | Matches: ❌
Proposed: E-X-P-S-E-T. | Actual: E-X-P-O-S-E | Matches: ❌
Proposed: F-S-R-E-C-O-S. | Actual: F-R-E-S-C-O-S | Matches: ❌
Proposed: W-I-P-S. | Actual: W-I-S-P | Matches: ❌
Proposed: M-I-R-E-G. | Actual: M-I-R-A-G-E | Matches: ❌
Proposed: I-V-O-R-Y. | Actual: I-V-O-R-Y | Matches: ❌
Proposed: O-N-S-H-O-R-D. | Actual: O-N-S-E-T | Matches: ❌
Proposed: E-L-E-U-D. | Actual: E-L-U-D-E | Matches: ❌
Proposed: S-P-H-I-N-X. | Actual: S-P-H-I-N-X | Matches: ❌
Proposed: B-R-A-N-Y. | Actual: B-R-A-W-N | Matches: ❌
Proposed: G-S-O-P-I-Y. | Actual: G-O-S-S-I-P-Y | Matches: ❌
Proposed: E-N-C-H-A-N-T. | Actual: E-N-C-H-A-N-T | Matches: ❌
Proposed: T-A-A-N-R. | Actual: T-A-V-E-R-N | Matches: ❌
Proposed: W-H-I-S-E. | Actual: W-H-I-S-T-L-E | Matches: ❌
Proposed: C-U-P-H-E-R-O-T. | Actual: C-A-P-T-U-R-E | Matches: ❌
Proposed: E-C-H-O-R-E. | Actual: E-C-H-O | Matches: ❌
Proposed: M-I-R-T.

The number of words has slightly increased. Let's try training using GRPO now.

First let's create some reward functions.

In [13]:
import re


def proportion_correct(word, proposed_spelling):
    correct_spelling = "-".join(word).upper()

    score = 0.0

    # Pad to the same length to handle extra characters
    max_len = max(len(correct_spelling), len(proposed_spelling))
    proposed_spelling_padded = proposed_spelling.ljust(max_len, " ")
    correct_spelling_padded = correct_spelling.ljust(max_len, " ")

    for a, b in zip(correct_spelling_padded, proposed_spelling_padded):
        if a == b:
            score += 1
        else:
            score -= 1
    return score / (
        len(correct_spelling)
    )  # Normalize by length of spelling, including dashes

assert proportion_correct("hello", "H-E-L-L-O") == 9 / 9
assert proportion_correct("hello", "H-E-L-") == 3 / 9
assert proportion_correct("hello", "H-E-L-L-O!") == 8 / 9

In [14]:
# Create a `reward_spelling` function that receives a batch of completions and the associated word values

import numpy as np


def reward_spelling(completions, word, **kwargs):
    """Reward function that rewards completions with more unique letters."""

    completion_strings = [
        completion.split("\n")[0].strip() for completion in completions
    ]
    words = [w for w in word]

    rewards = [proportion_correct(w, c) for w, c in zip(words, completion_strings)]


    print("=====")
    print(
        "Completion example first line:",
        words[0],
        "->",
        completion_strings[0].strip().split("\n")[0].strip(),
    )
    print(f"Spelling mean and std: {np.mean(rewards):.3f} +/- {np.std(rewards):.3f}")
    return rewards


assert reward_spelling(
    completions=[
        "H-E-L-L-O",
        "H-E-L-",
        "H-E-L-L-O!",
    ],
    word=[
        "hello",
        "hello",
        "hello",
    ],
) == [1, 3 / 9, 8 / 9]

=====
Completion example first line: hello -> H-E-L-L-O
Spelling mean and std: 0.741 +/- 0.292


In [15]:
# A reward of 1.0 for completions starting with a string formatted like X-Y-Z else return 0.0


def reward_response_in_form_of_letter_dash_letter(completions, word, **kwargs):
    """Reward function that gives a bonus for completions in the form of LETTER-DASH-LETTER."""
    pattern = re.compile(r"^([A-Z]-)+[A-Z]")  # Pattern for LETTER-DASH-LETTER

    words = [w for w in word]


    completion_strings = [
        completion.split("\n")[0].strip() for completion in completions
    ]

    rewards = [
        1.0 if pattern.match(c) else 0.0 for w, c in zip(words, completion_strings)
    ]


    print(
        f"Letter-dash-letter rewards mean and std: {np.mean(rewards):.3f} +/- {np.std(rewards):.3f}"
    )
    return rewards


assert reward_response_in_form_of_letter_dash_letter(
    completions=[
        "H-E-L-L-O",
        "hello",
        "Hi!",
    ],
    word=[
        "hello",
        "hello",
        "hello",
    ],
) == [1, 0, 0]

Letter-dash-letter rewards mean and std: 0.333 +/- 0.471


In [ ]:
# Set the GRPOConfig and initialize the trainer

from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    output_dir="data/spelling-grpo",
    max_completion_length=30,
    logging_steps=5,
    learning_rate=5e-5,
    num_train_epochs=10,  # We'll train just for a few epochs
    per_device_train_batch_size=8,  # The batch size for training
    num_generations=4,  # Determines the number of completions to compute for each single prompt
    lr_scheduler_type="cosine",
    beta=0.0,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[
        reward_spelling,
        reward_response_in_form_of_letter_dash_letter,
    ],
    args=training_args,
    train_dataset=ds["train"],
)
trainer.train()

## 6- Evaluate the fine-tuned model

In [17]:
# Evaluate the fine-tuned model on the same training examples

proportion_correct = 0.0

for example in ds["train"].select(range(20)):
    prompt = example["prompt"]
    completion = example["completion"]
    result = check_spelling(
        model=model,
        tokenizer=tokenizer,
        prompt=prompt,
        actual_spelling=completion,
        max_new_tokens=20,
    )
    proportion_correct += result

print(f"{proportion_correct}/20.0 words correct")

Proposed: T-I-R-U-M-P-H. | Actual: T-R-I-U-M-P-H. | Matches: ❌
Proposed: S-A-P-I-Z-R-H. | Actual: S-A-P-P-H-I-R-E. | Matches: ❌
Proposed: E-X-P-S-E. | Actual: E-X-P-O-S-E. | Matches: ❌
Proposed: F-R-S-E-C-O-S. | Actual: F-R-E-S-C-O-S. | Matches: ❌
Proposed: W-I-P-S. | Actual: W-I-S-P. | Matches: ❌
Proposed: M-I-R-G-E. | Actual: M-I-R-A-G-E. | Matches: ❌
Proposed: I-V-O-R-Y. | Actual: I-V-O-R-Y. | Matches: ✅
Proposed: O-N-S-H-E-D. | Actual: O-N-S-E-T. | Matches: ❌
Proposed: E-L-U-D-E. | Actual: E-L-U-D-E. | Matches: ✅
Proposed: S-P-H-I-N-X. | Actual: S-P-H-I-N-X. | Matches: ✅
Proposed: B-R-A-N-E. | Actual: B-R-A-W-N. | Matches: ❌
Proposed: G-O-S-H-O-P-I-Y. | Actual: G-O-S-S-I-P-Y. | Matches: ❌
Proposed: E-N-C-H-A-N. | Actual: E-N-C-H-A-N-T. | Matches: ❌
Proposed: T-A-A-N-R. | Actual: T-A-V-E-R-N. | Matches: ❌
Proposed: W-H-I-C-E. | Actual: W-H-I-S-T-L-E. | Matches: ❌
Proposed: C-U-P-H-R-E. | Actual: C-A-P-T-U-R-E. | Matches: ❌
Proposed: E-C-H-O-R-E. | Actual: E-C-H-O. | Matches: ❌
Propo

The model now performs better on the training data it has seen. But has it generalized? Let's check its performance on the unseen test set.

In [21]:
# Evaluate the fine-tuned model on the unseen test set

proportion_correct = 0.0
num_examples = len(ds["test"])

for example in ds["test"]:
    prompt = example["prompt"]
    completion = example["completion"]
    result = check_spelling(
        model=model,
        tokenizer=tokenizer,
        prompt=prompt,
        actual_spelling=completion,
        max_new_tokens=20,
    )
    proportion_correct += result

print(f"{proportion_correct}/{num_examples}.0 words correct")

Proposed: W-R-Y-I-L-Y. | Actual: W-R-Y-L-Y. | Matches: ❌
Proposed: G-I-L-N-E. | Actual: G-L-I-S-T-E-N. | Matches: ❌
Proposed: C-A-S-E-L. | Actual: Q-U-E-S-T. | Matches: ❌
Proposed: C-E-R-A-V. | Actual: C-R-A-V-E. | Matches: ❌
Proposed: L-U-S-I-R-O. | Actual: L-U-S-H. | Matches: ❌
Proposed: F-A-L-I-C-E. | Actual: F-A-B-L-E. | Matches: ❌
Proposed: K-N-A-R-C-E. | Actual: K-N-A-C-K. | Matches: ❌
2.5583333333333336/7.0 words correct


With a larger dataset and more training, it could get even better.